<a href="https://colab.research.google.com/github/aryastark08/KYBER-ICNN-Side-Channel-Analysis_Masked_Implementation/blob/main/Kyber_ICNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kyber-512 SCA done using Multi-bit Recovery ICNN

### Created by Aishwarya Nair

Run all the below commands one-by-one.

**DO NOT RUN ALL AT ONCE.**


In [1]:
!pip install h5py scalib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.4 MB/s eta 0:00:00


Run the following to download Kyber and all related dependencies

In [2]:
%cd /content

!git clone https://github.com/pq-crystals/kyber

%cd /content/kyber/ref
!make shared

# Verify libraries built
!ls -la lib/

%cd /content

/content
Cloning into 'kyber'...
remote: Enumerating objects: 2299, done.
remote: Counting objects: 100% (879/879), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 2299 (delta 778), reused 705 (delta 705), pack-reused 1420 (from 1)
Receiving objects: 100% (2299/2299), 952.00 KiB | 3.38 MiB/s, done.
Resolving deltas: 100% (1621/1621), done.
/content/kyber/ref
mkdir -p lib
cc -shared -fPIC -Wall -Wextra -Wpedantic -Wmissing-prototypes -Wredundant-decls -Wshadow -Wpointer-arith -O3 -fomit-frame-pointer -z noexecstack -DKYBER_K=2 kem.c indcpa.c polyvec.c poly.c ntt.c cbd.c reduce.c verify.c symmetric-shake.c -o lib/libpqcrystals_kyber512_ref.so
mkdir -p lib
cc -shared -fPIC -Wall -Wextra -Wpedantic -Wmissing-prototypes -Wredundant-decls -Wshadow -Wpointer-arith -O3 -fomit-frame-pointer -z noexecstack -DKYBER_K=3 kem.c indcpa.c polyvec.c poly.c ntt.c cbd.c reduce.c verify.c symmetric-shake.c -o lib/libpqcrystals_kyber768_ref.so
mkdir -p lib
cc -shared -fPIC -Wall -Wex

The following command loads the main Library "libpqcrystals_fips202_ref.so" globally and checks if everything is correct.

In [3]:
import os
import subprocess
import ctypes

# Build combined library WITH randombytes included
os.chdir('/content/kyber/ref')
os.system("""gcc -shared -fPIC -o /usr/lib/libpqcrystals_kyber512_ref.so \
    kem.c indcpa.c poly.c polyvec.c reduce.c ntt.c cbd.c \
    verify.c randombytes.c symmetric-shake.c fips202.c \
    -DKYBER_K=2 -I. -O2""")

# Copy fips202 separately
subprocess.run(['cp', '/content/kyber/ref/lib/libpqcrystals_fips202_ref.so', '/usr/lib/'], check=True)

# Update linker cache
subprocess.run(['ldconfig'], check=True)

# Set LD_LIBRARY_PATH
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

# Load libraries
fips  = ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
kyber = ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

print('fips202 loaded!')
print('Kyber loaded!')

os.chdir('/content')

fips202 loaded!
Kyber loaded!


Cloning the Dobias et al. Attack from Github Repo.

###IMP because it contains the kyber.py that contains the extract_msg function used by our ICNN.

In [4]:
%cd /content
!git clone https://github.com/Paprikadobi/Message-Recovery-Attack-on-ML-KEM.git Message-Recovery-Attack-on-ML-KEM
!ls /content/Message-Recovery-Attack-on-ML-KEM/

/content
Cloning into 'Message-Recovery-Attack-on-ML-KEM'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 3), reused 18 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 6.19 KiB | 6.19 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Filtering content: 100% (4/4), 1.64 GiB | 13.39 MiB/s, done.
datasets  kyber.py  main.py  models.py	Readme.md  requirements.txt  utils.py


In [5]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU name       : {torch.cuda.get_device_name(0)}")
print(f"GPU RAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

CUDA available : True
GPU name       : Tesla T4
GPU RAM        : 14.6 GB


#### Checking if the kyber.py is accessible

In [6]:
%cd /content/Message-Recovery-Attack-on-ML-KEM

import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

import ctypes
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

from kyber import extract_msg
import h5py

print('Testing extract_msg...')
with h5py.File('datasets/kem_dec_unprotected_8.h5', 'r') as f:
    inputs = f['inputs']
    for i in range(3):
        data = inputs[i]
        sk   = bytes(data[:1632])
        c    = bytes(data[1632:2400])
        msg  = extract_msg(sk, c)
        print(f'Sample {i}: msg[:4] = {list(msg[:4])}')

print('\nextract_msg working!')

/content/Message-Recovery-Attack-on-ML-KEM
Testing extract_msg...
Sample 0: msg[:4] = [np.uint16(104), np.uint16(70), np.uint16(128), np.uint16(55)]
Sample 1: msg[:4] = [np.uint16(139), np.uint16(193), np.uint16(26), np.uint16(167)]
Sample 2: msg[:4] = [np.uint16(36), np.uint16(203), np.uint16(124), np.uint16(68)]

extract_msg working!


#### Checking if the GPU is running and not CPU

In [7]:
import torch
print(torch.cuda.is_available())        # Should print: True
print(torch.cuda.get_device_name(0))    # Should print: Tesla T4

True
Tesla T4


In [ ]:
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments

# Re-set LD_LIBRARY_PATH for this cell
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

!python big_icnn_train.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device: cuda
[*] GPU: Tesla T4
[*] GPU RAM: 14.6 GB

[+] Loaded CNN-optimized POIs : (32, 534)
[*] Max POI index             : 5999 (must be < 6000)
[*] Samples per channel       : 534
[*] Prediction type           : Binary (32 bytes x 8 bits = 256 bits)
[*] Trace limit               : 60000
[*] Epochs                    : 50

[*] Estimated RAM usage:
    X_tensor : 3.82 GB
    y_tensor : 0.11 GB
    Total    : 3.93 GB (data on CPU RAM)

[*] Checking model size...
[+] Trainable parameters : 17,895,936
[+] Model RAM estimate   : 0.20 GB

[*] Loading dataset (limit=60000)...
[*] Pre-loading 60000 traces into CPU RAM...
    -> Cached 10000/60000...
    -> Cached 20000/60000...
    -> Cached 30000/60000...
    -> Cached 40000/60000...
    -> Cached 50000/60000...
[+] X_tensor : torch.Size([60000, 32, 534])
[+] y_tensor : torch.Size([60000, 32, 8])

[*] Initializing InterconnectedKyberCNN...
[+] Trainable parameters: 17,8

# Proof-Of-Concept Single-Bit Model Training and Testing Script

We train the model and subsequently test it for all bits [0-7] on the Attack dataset, giving us a detailed analysis of per-bit accuracy of our model in recovering the message in 1 trace and when averaged.

In [8]:
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments

# Re-set LD_LIBRARY_PATH for this cell
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

!python train_attack_all_bits.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device : cuda
[*] Will train and attack bits: [0, 1, 2, 3, 4, 5, 6, 7]
[+] CNN-optimized POIs: (32, 534)

[*] Pre-loading attack dataset...
[+] Attack: 200 secrets × 256 traces

  TRAINING BIT 0
A. Entering Dataset Init
B. HDF5 Opened
C. Traces loaded
D. Inputs loaded
E. Using POIs for byte 0: 534 points
F. Dataset size: 40000 traces
[+] Labels: 0=20045 | 1=19955
  Epoch 01/25 | Acc: 49.97% ↑ | Best: 49.97%
  Epoch 02/25 | Acc: 49.90%   | Best: 49.97%
  Epoch 03/25 | Acc: 49.88%   | Best: 49.97%
  Epoch 04/25 | Acc: 50.07% ↑ | Best: 50.07%
  Epoch 05/25 | Acc: 50.06%   | Best: 50.07%
  Epoch 06/25 | Acc: 50.20% ↑ | Best: 50.20%
  Epoch 07/25 | Acc: 50.05%   | Best: 50.20%
  Epoch 08/25 | Acc: 50.12%   | Best: 50.20%
  Epoch 09/25 | Acc: 50.52% ↑ | Best: 50.52%
  Epoch 10/25 | Acc: 50.44%   | Best: 50.52%
  Epoch 11/25 | Acc: 50.37%   | Best: 50.52%
  Epoch 12/25 | Acc: 50.73% ↑ | Best: 50.73%
  Epoch 13/25 | Acc: 50.

# Main Multi-bit Model Training Script

In [ ]:
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments

# Re-set LD_LIBRARY_PATH for this cell
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

!python big_icnn_train_v2.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device: cuda
[*] GPU: Tesla T4
[*] GPU RAM: 14.6 GB

[+] CNN-optimized POIs  : (32, 534)
[*] Max POI index       : 5999
[*] Samples per channel : 534
[*] Trace limit         : 90000
[*] Learning rate       : 0.0001
[*] Epochs              : 50

[*] RAM: X=5.73GB | y=0.17GB | Total=5.90GB
[*] Parameters: 17,895,936

[*] Loading dataset...
[*] Pre-allocating tensors (no peak RAM spike)...
    -> 10000/90000...
    -> 20000/90000...
    -> 30000/90000...
    -> 40000/90000...
    -> 50000/90000...
    -> 60000/90000...
    -> 70000/90000...
    -> 80000/90000...
[+] X_tensor : torch.Size([90000, 32, 534])
[+] y_tensor : torch.Size([90000, 32, 8])

[*] Initializing model...
[+] Parameters: 17,895,936
[+] Output shape: torch.Size([2, 32, 8, 2]) ✓

[*] Training (LR=0.0001, traces=90000)...

Epoch 01/50 | Loss: 242.5397 | Train Acc: 53.10% ↑ | Best: 53.10%
Epoch 02/50 | Loss: 236.1971 | Train Acc: 58.39% ↑ | Best: 58.39%
Ep

# Main multi-bit model Attack executed on the Test dataset

The crucial result here is that the model achieves a 60.25% accuracy in recovering the message with a single trace N=1.

In [ ]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python big_icnn_attack_v2.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device : cuda
[+] POI shape           : (32, 534)

[*] Loading attack dataset...
[+] Total traces        : 51,200
[+] Unique secrets      : 200
[+] Traces per secret   : 256

[*] Computing true labels...
[+] Labels shape        : (200, 32, 8)

[*] Loading model weights: big_icnn_best.pt
[+] Parameters          : 17,895,936
[+] Model ready!

[*] Running attack across averaging levels...

  N=  1 | Bit acc: 60.25% | vs random: +10.25%
  N=  2 | Bit acc: 60.00% | vs random: +10.00%
  N=  4 | Bit acc: 59.59% | vs random: +9.59%
  N=  8 | Bit acc: 58.79% | vs random: +8.79%
  N= 16 | Bit acc: 57.77% | vs random: +7.77%
  N= 32 | Bit acc: 56.50% | vs random: +6.50%
  N= 64 | Bit acc: 54.64% | vs random: +4.64%
  N=128 | Bit acc: 52.95% | vs random: +2.95%
  N=256 | Bit acc: 50.03% | vs random: +0.03%

  BEST: N=1 → 60.25% overall bit accuracy

Per-byte breakdown at N=1:
  Byte |  Bit Acc |   vs 50%
------------------------

## Single-bit training Script


### Single bit training script for bit index = 1


In [9]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python train_cnn.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device: cuda
[+] Using CNN-optimized POIs: (32, 534)
[*] Recovering bit index : 1
[*] Trace limit          : 90000

[*] Creating dataset...
A. Entering Dataset Init
B. HDF5 Opened
C. Traces loaded
D. Inputs loaded
E. Using POIs for byte 1: 534 points
F. Dataset size: 90000 traces
[*] Pre-loading 90000 traces into RAM...
    -> Loaded 5000/90000 traces...
    -> Loaded 10000/90000 traces...
    -> Loaded 15000/90000 traces...
    -> Loaded 20000/90000 traces...
    -> Loaded 25000/90000 traces...
    -> Loaded 30000/90000 traces...
    -> Loaded 35000/90000 traces...
    -> Loaded 40000/90000 traces...
    -> Loaded 45000/90000 traces...
    -> Loaded 50000/90000 traces...
    -> Loaded 55000/90000 traces...
    -> Loaded 60000/90000 traces...
    -> Loaded 65000/90000 traces...
    -> Loaded 70000/90000 traces...
    -> Loaded 75000/90000 traces...
    -> Loaded 80000/90000 traces...
    -> Loaded 85000/90000 traces.

### Single bit training script for bit index = 2

In [10]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python train_cnn.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Using device: cuda
[+] Using CNN-optimized POIs: (32, 534)
[*] Recovering bit index : 2
[*] Trace limit          : 90000

[*] Creating dataset...
A. Entering Dataset Init
B. HDF5 Opened
C. Traces loaded
D. Inputs loaded
E. Using POIs for byte 2: 534 points
F. Dataset size: 90000 traces
[*] Pre-loading 90000 traces into RAM...
    -> Loaded 5000/90000 traces...
    -> Loaded 10000/90000 traces...
    -> Loaded 15000/90000 traces...
    -> Loaded 20000/90000 traces...
    -> Loaded 25000/90000 traces...
    -> Loaded 30000/90000 traces...
    -> Loaded 35000/90000 traces...
    -> Loaded 40000/90000 traces...
    -> Loaded 45000/90000 traces...
    -> Loaded 50000/90000 traces...
    -> Loaded 55000/90000 traces...
    -> Loaded 60000/90000 traces...
    -> Loaded 65000/90000 traces...
    -> Loaded 70000/90000 traces...
    -> Loaded 75000/90000 traces...
    -> Loaded 80000/90000 traces...
    -> Loaded 85000/90000 traces.

### This is a plot for the single training script that describes Byte-wise leakage differences.

In [9]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python plot_single_bit_results.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[+] Saved: single_bit_comparison.pdf and single_bit_comparison.png
Figure(1500x750)


In [10]:
from google.colab import files

files.download('single_bit_comparison.pdf')
files.download('single_bit_comparison.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Plot for the POI Correlations


In [11]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python new_poi.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[*] Finding strongest signal regions...

Top 10 most correlated time points (byte 0):
  Rank  1: time=5944 | correlation=0.074697
  Rank  2: time= 549 | correlation=0.071489
  Rank  3: time= 532 | correlation=0.068176
  Rank  4: time=  40 | correlation=0.067594
  Rank  5: time= 883 | correlation=0.067131
  Rank  6: time=5372 | correlation=0.066194
  Rank  7: time=3448 | correlation=0.064683
  Rank  8: time= 553 | correlation=0.063973
  Rank  9: time=3177 | correlation=0.063966
  Rank 10: time=4294 | correlation=0.062061

Current POI range 1: 233-633
Current POI range 2: 3367-3499
[+] Saved: correlation_profile.npy

Correlation analysis:
  Overall mean    : 0.020831
  POI region 1    : 0.024195
  POI region 2    : 0.020904
  Best time point : 0.074697


In [ ]:
files.download('single_bit_comparison.png')

# POI Comparison between LDA and CNN

In [14]:
# Make sure libraries are loaded first!
import os, ctypes
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL('/usr/lib/libpqcrystals_fips202_ref.so', mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL('/usr/lib/libpqcrystals_kyber512_ref.so')

# Then run
%cd /content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
!python plot_poi_comparison.py

/content/Message-Recovery-Attack-on-ML-KEM/icnn_experiments
[+] Saved: poi_comparison.pdf and poi_comparison.png
Figure(1800x1200)


In [15]:
from google.colab import files

files.download('poi_comparison.pdf')
files.download('poi_comparison.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')